# Neural Hydrology Playground — exploration notebook

This notebook walks through the whole learning workflow on real data:

1. **Phase 1 – look at the data**: one basin's weather (dynamic inputs), its static attributes and the observed streamflow (target).
2. **Phase 2/3 – model results**: observed vs predicted hydrographs, NSE / KGE, LSTM vs Transformer.
3. A tiny **flood-threshold** demonstration.

Run `python scripts/download_data.py` first (and `python scripts/train.py && python scripts/evaluate.py && python scripts/compare.py`
for the model sections). Every number printed here comes from the files under `data/` and `outputs/`.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from nhplay.data.camels import (DYNAMIC_INPUTS, STATIC_ATTRIBUTES, TARGET, load_basin, load_attributes,
                                load_discharge_cfs, basin_summary, read_basin_list)
from nhplay.evaluation import metrics as M
from nhplay.visualization import plots
from nhplay.paths import PREDICTION_DIR, METRIC_DIR

%matplotlib inline
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
basins = read_basin_list()
basins

## Phase 1 · the six basins and their static attributes

One row per basin, constant in time. These are the model's **static inputs**.

In [ ]:
attrs = load_attributes(basins)
attrs[["gauge_name", "area_gages2", "elev_mean", "slope_mean", "frac_forest", "aridity", "frac_snow", "p_mean", "soil_depth_pelletier"]].round(2)

### Which columns are inputs and which is the target?

In [ ]:
bd = load_basin("01022500")   # Narraguagus River at Cherryfield, Maine
bd.variable_table()

### Dynamic inputs + target: the daily table for one basin

Each row is one day. The first five columns are **weather** (what the model sees); the last one is **observed streamflow** (what it must predict).

In [ ]:
ts = bd.timeseries[list(DYNAMIC_INPUTS) + ["QObs(mm/d)"]]
print(ts.shape, "days x columns ;", ts.index.min().date(), "to", ts.index.max().date())
ts.loc["2010-09-28":"2010-10-08"].round(2)

Read the table: 26.4 mm of rain fell on 2 Oct 2010. The river rose from 0.32 to 1.31 mm/day the same day, peaked at
1.54 mm/day on 3 Oct (one day *after* the rain) and then drained slowly over the following days. That shape — rising limb, peak,
recession — is the basic unit of a **hydrograph**.

### The unit conversion: from cubic feet per second at the gauge to mm/day over the basin

In [ ]:
cfs = load_discharge_cfs("01022500")
area_m2 = bd.area_m2
demo = pd.DataFrame({"QObs(cfs)": cfs.loc["2010-10-01":"2010-10-04"]})
demo["QObs(mm/d) by hand"] = demo["QObs(cfs)"] * 0.0283168 * 86400 / area_m2 * 1000
demo["QObs(mm/d) NeuralHydrology"] = bd.timeseries.loc["2010-10-01":"2010-10-04", "QObs(mm/d)"]
print(f"basin area = {area_m2/1e6:.1f} km^2")
demo.round(3)

### Plots: rain, temperature and streamflow for one water year, then the full record

In [ ]:
fig = plots.plot_forcing_and_flow(bd.timeseries, "01022500 Narraguagus River — water year 2011", start="2010-10-01", end="2011-09-30")
plt.show()

In [ ]:
fig = plots.plot_forcing_and_flow(bd.timeseries, "01022500 Narraguagus River — full record 1980-2014")
plt.show()

### Water balance: how much of the rain leaves as river flow?

Because both are in mm/day we can compare them directly. The **runoff ratio** varies from 15 % (hot, dry Texas) to 97 % (wet, cool Pacific coast).

In [ ]:
rows = []
for b in basins:
    d = load_basin(b)
    t = d.timeseries
    rows.append({"basin": b, "name": d.attributes["gauge_name"], "mean rain mm/day": t["prcp(mm/day)"].mean(),
                 "mean flow mm/day": t["QObs(mm/d)"].mean(), "runoff ratio": t["QObs(mm/d)"].mean() / t["prcp(mm/day)"].mean(),
                 "max flow mm/day": t["QObs(mm/d)"].max(), "date of max": t["QObs(mm/d)"].idxmax().date()})
pd.DataFrame(rows).round(2)

### Why the past matters: the same rain on a dry vs a wet basin (Blanco River, Texas)

In [ ]:
t = load_basin("08171300").timeseries
top = t["QObs(mm/d)"].nlargest(5)
pd.DataFrame({
    "flow mm/day": top,
    "rain that day": [t.loc[d, "prcp(mm/day)"] for d in top.index],
    "rain previous 3 days": [t.loc[d - pd.Timedelta(days=3): d - pd.Timedelta(days=1), "prcp(mm/day)"].sum() for d in top.index],
    "flow the day before": [t.loc[d - pd.Timedelta(days=1), "QObs(mm/d)"] for d in top.index],
}).round(1)

### Snow: rain in December, river in June (South Fork Payette, Idaho)

In [ ]:
t = load_basin("13235000").timeseries
t.loc["2010-10-01":"2011-09-30", ["prcp(mm/day)", "tmax(C)", "QObs(mm/d)"]].resample("MS").mean().round(2)

## Phase 2/3 · model results (LSTM vs Transformer)

The cells below read `outputs/predictions/*.csv` written by `scripts/evaluate.py`. Both models were trained on the same six basins,
the same inputs and target, and the same chronological split (train 1980-1999, validation 1999-2004, **test 2004-2014**).

In [ ]:
preds = {}
for m in ("lstm", "transformer"):
    f = PREDICTION_DIR / f"{m}_test.csv"
    if f.exists():
        preds[m] = pd.read_csv(f, dtype={"basin": str}, parse_dates=["date"])
print("models with predictions:", list(preds))
if not preds:
    print("Run: python scripts/train.py && python scripts/evaluate.py && python scripts/compare.py")

In [ ]:
if preds:
    rows = []
    for m, p in preds.items():
        for b, g in p.groupby("basin"):
            sc = M.scorecard(g["obs"].values, g["sim"].values).as_dict()
            rows.append({"basin": b, "model": m, **sc})
    table = pd.DataFrame(rows)
    display(table.pivot(index="basin", columns="model", values=["nse", "kge"]).round(3))
    display(table.groupby("model")[["nse", "kge", "r", "alpha", "beta", "fhv"]].median().round(3))

### Hydrographs: observed vs predicted for one water year of the test period

In [ ]:
if preds:
    basin = "01022500"
    d = load_basin(basin)
    obs = preds[next(iter(preds))].query("basin == @basin").set_index("date")["obs"]
    sims = {m: p.query("basin == @basin").set_index("date")["sim"] for m, p in preds.items()}
    fig = plots.plot_hydrograph(obs, sims, f"{basin} {d.attributes['gauge_name']} — water year 2011 (test period)",
                                rain=d.timeseries["prcp(mm/day)"], start="2010-10-01", end="2011-09-30")
    plt.show()

In [ ]:
if preds:
    fig = plots.plot_scatter(obs, sims, f"{basin} observed vs predicted, test period 2004-2014"); plt.show()
    fig = plots.plot_flow_duration_curve(obs, sims, f"{basin} flow duration curve, test period"); plt.show()

### The biggest observed flood of the test period in each basin, and what each model said that day

In [ ]:
if preds:
    rows = []
    for b in basins:
        d = load_basin(b)
        o = preds[next(iter(preds))].query("basin == @b").set_index("date")["obs"]
        day = o.idxmax()
        row = {"basin": b, "peak date": day.date(), "rain that day mm": d.timeseries.loc[day, "prcp(mm/day)"], "observed mm/day": o.max()}
        for m, p in preds.items():
            row[m + " mm/day"] = p.query("basin == @b").set_index("date").loc[day, "sim"]
        rows.append(row)
    display(pd.DataFrame(rows).round(1))

## A tiny flood-threshold demonstration (educational only)

Operational systems compare forecasts with return-period thresholds (e.g. the 2-year flood). We have no official flood stages for
these gauges, so we simply use the 99th percentile of each basin's observed daily flow and count exceedance days in the test period.

In [ ]:
if preds:
    rows = []
    for b in basins:
        d = load_basin(b)
        thr = np.nanpercentile(d.timeseries["QObs(mm/d)"], 99)
        o = preds[next(iter(preds))].query("basin == @b").set_index("date")["obs"]
        row = {"basin": b, "99th pct threshold mm/day": thr, "observed days above": int((o > thr).sum())}
        for m, p in preds.items():
            s = p.query("basin == @b").set_index("date")["sim"]
            row[m + " days above"] = int((s > thr).sum())
            row[m + " hits"] = int(((s > thr) & (o > thr)).sum())
        rows.append(row)
    display(pd.DataFrame(rows).round(1))

## Where to go next

* Change `data/basins.txt`, re-run the download and training, and see how the models cope with a new climate.
* Train a single-basin LSTM (edit `train_basin_file`) and compare it with the multi-basin one.
* Try `transformer_nlayers: 4` or a larger `hidden_size` — but keep the validation period for model selection and never tune on the test period.
* Read `LEARNING_NOTES.md` for the plain-English background and `README.md` for the research extensions.